In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pyspark.sql.functions as F

from src.datasets import read_qqp
from src.spark_utils import get_spark

%load_ext autoreload
%autoreload 2

spark = get_spark()

In [ ]:
import math
import time

from src.datasets import read_synthetic
from src.near_duplicates import detect_near_duplicate_batch

df = read_synthetic()
dataset, query = df.randomSplit([0.8, 0.2])
df.write.format("noop").mode("overwrite").save()
query.write.format("noop").mode("overwrite").save()
dataset.write.format("noop").mode("overwrite").save()
print(f"Size of dataframes - dataset: {dataset.count()}, query: {query.count()}")
start = time.perf_counter()
duplicates = detect_near_duplicate_batch(
    dataset, query, 64, 3, int(math.log(dataset.count(), 2)), 10
)
duplicates.write.format("noop").mode("overwrite").save()
end = time.perf_counter()
print(f"Time to execute: {(end - start) * 1000}")

In [ ]:
import re
from collections import Counter


def find_repeated_words(text: str) -> list[str]:
    # Extract only alphanumeric words and convert them to lowercase
    words = re.findall(r"\b\w+\b", text.lower())

    # Count occurrences of each word
    word_counts = Counter(words)

    # Filter words appearing at least twice
    repeated = [word for word, count in word_counts.items() if count >= 2]

    return repeated


if __name__ == "__main__":
    sample_text = "Don't worry about people stealing your ideas. If your ideas are any good, you'll have to ram them down people's throats."

    result = find_repeated_words(sample_text)
    print("Repeated words:", result)

In [ ]:
import time

import numpy as np

from src.simhash import _numba_fastsimhash

rng = np.random.default_rng()
arr = rng.integers(0, 256, size=(1_000_000, 8), dtype=np.uint8)
n_hashes = 1_000_000
hash_len = 8


def simhash(byte_matrix: np.ndarray, n_hashes: int, hash_len: int) -> np.ndarray:
    counters = np.zeros(hash_len * 8, dtype=np.int64)

    for j in range(hash_len):
        base_idx = j * 8
        for i in range(n_hashes):
            val = byte_matrix[i, j]  # val is a byte
            counters[base_idx] += (val >> 7) & 1
            counters[base_idx + 1] += (val >> 6) & 1
            counters[base_idx + 2] += (val >> 5) & 1
            counters[base_idx + 3] += (val >> 4) & 1
            counters[base_idx + 4] += (val >> 3) & 1
            counters[base_idx + 5] += (val >> 2) & 1
            counters[base_idx + 6] += (val >> 1) & 1
            counters[base_idx + 7] += val & 1

    output = np.zeros(hash_len, dtype=np.uint8)
    threshold = (n_hashes + 1) // 2

    for i in range(hash_len):
        base_idx = i * 8
        part_sum = 0
        if counters[base_idx] >= threshold:
            part_sum |= 128
        if counters[base_idx + 1] >= threshold:
            part_sum |= 64
        if counters[base_idx + 2] >= threshold:
            part_sum |= 32
        if counters[base_idx + 3] >= threshold:
            part_sum |= 16
        if counters[base_idx + 4] >= threshold:
            part_sum |= 8
        if counters[base_idx + 5] >= threshold:
            part_sum |= 4
        if counters[base_idx + 6] >= threshold:
            part_sum |= 2
        if counters[base_idx + 7] >= threshold:
            part_sum |= 1
        output[i] = part_sum

    return output


print("--- Comparison between SimHash and FastSimHash ---")
print("Round 1:")
start = time.perf_counter()
result = simhash(arr, n_hashes, hash_len)
end = time.perf_counter()
print(f"SimHash time: {(end - start) * 1000} ms")
start = time.perf_counter()
result = _numba_fastsimhash(arr, n_hashes, hash_len)
end = time.perf_counter()
print(f"FastSimHash time: {(end - start) * 1000} ms")
print("Round 2:")
start = time.perf_counter()
result = simhash(arr, n_hashes, hash_len)
end = time.perf_counter()
print(f"SimHash time: {(end - start) * 1000} ms")
start = time.perf_counter()
result = _numba_fastsimhash(arr, n_hashes, hash_len)
end = time.perf_counter()
print(f"FastSimHash time: {(end - start) * 1000} ms")

In [ ]:
import numpy as np

dummy_matrix = np.zeros((10, 64), dtype=np.uint8)

_numba_fastsimhash(dummy_matrix, 10, 64)

llvm_dict = _numba_fastsimhash.inspect_llvm()

for signature, llvm_ir in llvm_dict.items():
    print(llvm_ir)  # look for <8 x i32>

In [ ]:
from src.datasets import read_c5
from src.fingerprint import create_fingerprints

df = read_c5()
df = df.withColumn("length", F.length("data"))
df.show()

In [ ]:
from src.datasets import read_c5

df = read_c5()
fp_df = create_fingerprints(
    df=df,
    model="embeddings",
    input_col="data",
    output_col="fingerprint",
    fingerprint_size=128,
    other=None,
)
fp_df.show()

In [ ]:
from src.fingerprint import create_fingerprints

df = read_qqp().limit(1000)
df = create_fingerprints(df, "embeddings", "data1", "fingerprint1", 64, None)
df = create_fingerprints(df, "embeddings", "data2", "fingerprint2", 64, None)
df.show()
# metrics = detect_near_duplicates_benchmark(df, "fingerprint1", "fingerprint2", "is_near_duplicate", 5)

In [ ]:
from src.near_duplicates import partitions_schema

f = 64
p_min = 10
K = 6
d = 20

partitions_schema(f=f, K=K, d=d, p_min=p_min)

In [ ]:
from pyspark.sql.types import BinaryType, IntegerType, StructField, StructType

from src.near_duplicates import detect_near_duplicate_v2

schema = StructType(
    [
        StructField("ID", IntegerType(), nullable=False),
        StructField("fingerprint", BinaryType(), nullable=False),
    ]
)

data1 = [
    (1, bytes.fromhex("0123456789ABCDEF")),
    (2, bytes.fromhex("00000000FFFFFFFF")),
    (3, bytes.fromhex("FFFFFFFF00000000")),
]

# Dati per il secondo DataFrame
data2 = [
    (10, bytes.fromhex("00000111FFFFFFFF")),
    (20, bytes.fromhex("8877000044332211")),
    (30, bytes.fromhex("CAFEBABE00112233")),
]

df1 = spark.createDataFrame(data=data1, schema=schema)
df2 = spark.createDataFrame(data=data2, schema=schema)

detect_near_duplicate_v2(df1, df2, 64, 3, "ID", "fingerprint", p_min=10, corpus_size=3).show()

In [ ]:
from src.datasets import read_c5, read_wikipedia

df = read_qqp()
df1 = df.filter(F.col("is_near_duplicate") == F.lit(True)).limit(3)
df2 = df.filter(F.col("is_near_duplicate") == F.lit(False)).limit(3)
df = df1.union(df2)
df.toPandas().to_csv("qqp_sample.csv")

In [ ]:
df = read_wikipedia(variant="10percent")
print(df.rdd.getNumPartitions())

df_limited = df.limit(300)
print(df_limited.rdd.getNumPartitions())

In [ ]:
spark.stop()